<a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/workflow_agent_sdk_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Data Analytics: Scheduled Workflow Agents

| Author |
| --- |
| [Geon Yoon](https://github.com/geonyoon) |

# Background and Overview

A workflow agent runs a Conversational Analytics question on a schedule and
delivers the result without anyone being present. This notebook creates one,
lists the ones you own, triggers a run on demand, reads the output, and cleans
up.

A workflow agent is a variant of `DataAgent`. It wraps an existing analytics
agent, adds a trigger, and records who created it. Two things about it differ
from an interactive agent and are worth understanding before you run anything
below.

**It runs as you, while you are not there.** Each run executes under the
creator's credentials, so it sees exactly the data that person can see and no
more. That requires a one-time consent grant. The first trigger below may fail
with an authorization error carrying a URL. Open it once, approve, and later
runs proceed without prompting.

**Only the creator can trigger it.** Holding project-level IAM on the agent is
not enough to run someone else's workflow, because the run would execute under
their identity and return their data.

Each run produces a `Conversation` titled
`"<workflow display name> - <run time>"`, which is how a client with no UI
tells one run from another. Give the workflow a `display_name` you can match
on, otherwise that title starts with an empty string.

Workflow agents are available in the `v1beta` and `v1alpha` channels. This
notebook uses `v1beta`.

# Get Started

## API Enablement

Enable the Gemini Data Analytics API in your project before running the cells
below. You also need an existing analytics agent for the workflow to execute.
Create one with the
[intro notebook](https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/intro_gemini_data_analytics_sdk.ipynb)
if you do not have one.

## Install the client library

In [ ]:
%pip install google-cloud-geminidataanalytics

# Setup

In [ ]:
# @title Authenticate
from google.colab import auth

auth.authenticate_user()

In [ ]:
# @title Import the Gemini Data Analytics client library
from google.cloud import geminidataanalytics_v1beta as geminidataanalytics

data_agent_client = geminidataanalytics.DataAgentServiceClient()
data_chat_client = geminidataanalytics.DataChatServiceClient()

In [ ]:
# @title Project, location and the agent to execute
# fmt: off
billing_project = ""  # @param {type:"string"}
location = "global"  # @param {type:"string"}

# The analytics agent the workflow will ask its question of.
execution_agent_id = ""  # @param {type:"string"}
# fmt: on

parent = f"projects/{billing_project}/locations/{location}"
execution_data_agent = f"{parent}/dataAgents/{execution_agent_id}"

# 1. Create a workflow agent

A workflow agent needs three things:

*   a **trigger**, here a cron expression plus an IANA time zone,
*   a **directive**, the question to ask on every run,
*   the **agent to execute** it against.

Destinations are optional. Results always land in a conversation; adding an
email destination also sends them on.

`CreateDataAgent` is a long-running operation, so poll it to completion.

In [ ]:
# @title 1. Create a workflow agent
# fmt: off
workflow_agent_id = "weekly_active_users"  # @param {type:"string"}
# Run conversations are titled "<display_name> - <run time>", so set something
# you can recognise.
display_name = "Weekly active users"  # @param {type:"string"}
question = "Calculate weekly active users and compare to the previous 4 weeks."  # @param {type:"string"}
crontab = "0 8 * * 1"  # @param {type:"string"}
time_zone = "America/Los_Angeles"  # @param {type:"string"}
email_recipients = ""  # @param {type:"string"}
# fmt: on

workflow_agent = geminidataanalytics.WorkflowAgent(
    execution_data_agent=execution_data_agent,
    triggers=[
        geminidataanalytics.Trigger(
            cron_trigger=geminidataanalytics.CronTrigger(
                crontab=crontab,
                time_zone=time_zone,
            )
        )
    ],
    directive=geminidataanalytics.Directive(
        follow_instructions_directive=(
            geminidataanalytics.FollowInstructionsDirective(
                instructions=[
                    geminidataanalytics.Instruction(natural_language=question)
                ]
            )
        )
    ),
)

recipients = [e.strip() for e in email_recipients.split(",") if e.strip()]
if recipients:
  workflow_agent.destinations = [
      geminidataanalytics.Destination(
          email_destination=geminidataanalytics.EmailDestination(
              recipients=recipients
          )
      )
  ]

request = geminidataanalytics.CreateDataAgentRequest(
    parent=parent,
    data_agent_id=workflow_agent_id,
    data_agent=geminidataanalytics.DataAgent(
        display_name=display_name,
        workflow_agent=workflow_agent,
    ),
)

created = data_agent_client.create_data_agent(request=request).result()
print(created)

# 2. List the workflows you created

`ListDataAgents` returns every agent you can see in the project, analytics and
workflow alike. Pass `creator_filter=CREATOR_ONLY` to narrow it to the ones you
own, which is almost always what a client with no UI wants, then keep the ones
that carry a `workflow_agent`.

In [ ]:
# @title 2. List your workflow agents
CreatorFilter = (
    geminidataanalytics.ListAccessibleDataAgentsRequest.CreatorFilter
)

request = geminidataanalytics.ListDataAgentsRequest(
    parent=parent,
    creator_filter=CreatorFilter.CREATOR_ONLY,
)

for agent in data_agent_client.list_data_agents(request=request):
  if "workflow_agent" in agent:
    print(agent.name)
    for trigger in agent.workflow_agent.triggers:
      print(
          f"  cron: {trigger.cron_trigger.crontab}"
          f" ({trigger.cron_trigger.time_zone})"
      )

# 3. Trigger a run on demand

Runs normally happen on the schedule. To start one immediately, send a chat
message carrying `workflow_params` with the `EXECUTE` action. The message text
is not the question; the question lives in the workflow's directive.

This is the call that surfaces the one-time consent requirement. If it fails
with an authorization error, open the URL in the error details, approve, and
run the cell again.

In [ ]:
# @title 3. Trigger a run
workflow_agent_name = f"{parent}/dataAgents/{workflow_agent_id}"

messages = [
    geminidataanalytics.Message(
        user_message=geminidataanalytics.UserMessage(
            text="Run this workflow now.",
            runtime_params=geminidataanalytics.RuntimeParams(
                workflow_params=geminidataanalytics.WorkflowParams(
                    workflow_agent=workflow_agent_name,
                    action=geminidataanalytics.WorkflowParams.Action.EXECUTE,
                )
            ),
        )
    )
]

request = geminidataanalytics.ChatRequest(
    parent=parent,
    messages=messages,
    data_agent_context=geminidataanalytics.DataAgentContext(
        data_agent=workflow_agent_name
    ),
)

for response in data_chat_client.chat(request=request):
  print(response)

# 4. Read the results of a run

Each run produces a `Conversation` titled `"<display name> - <run time>"`. List
conversations, keep the ones belonging to this workflow, then read their
messages.

In [ ]:
# @title 4. Read run output
request = geminidataanalytics.ListConversationsRequest(parent=parent)

run_conversations = [
    c
    for c in data_chat_client.list_conversations(request=request)
    if c.title.startswith(f"{display_name} - ")
]

for conversation in run_conversations:
  print(f"=== {conversation.title} ({conversation.name}) ===")
  messages = data_chat_client.list_messages(
      request=geminidataanalytics.ListMessagesRequest(parent=conversation.name)
  )
  for message in messages:
    print(message)

# 5. Cleanup

Deleting the workflow agent also cancels its schedule. Conversations produced by
past runs are not deleted with it; remove them separately if you want to.

In [ ]:
# @title 5. Delete the workflow agent
request = geminidataanalytics.DeleteDataAgentRequest(
    name=f"{parent}/dataAgents/{workflow_agent_id}"
)

data_agent_client.delete_data_agent(request=request).result()
print("deleted")